In [ ]:
import pandas as pd

#  Load the dataset
df = pd.read_csv("cleaned_data.csv")

#  Define Regression label
# Using 'TotalCharges' as the continuous numeric target
y_reg = df['TotalCharges']

# Define Classification label
# Using the natural binary column 'Churn' (Mapping 'yes' to 1 and 'no' to 0)
# .str.strip() removes hidden spaces, .str.lower() makes everything lowercase
y_clf = (df['Churn'].astype(str).str.strip().str.lower() == 'yes').astype(int)

# Verify that you now have both 0s and 1s!
print(y_clf.value_counts())
# Note: If you prefer to binarize the regression label at its median instead, use:
#y_clf = (y_reg > y_reg.median()).astype(int)

# 4. Define Feature matrix X
# Dropping the target variables and the unique customerID string
X = df.drop(columns=['TotalCharges', 'Churn', 'customerID'])

In [52]:
# 1. Label Encoding for Ordinal Categories
# Defining the natural order for the 'Contract' column
contract_mapping = {
    'month-to-month': 0,
    'one year': 1,
    'two year': 2
}
X['Contract'] = X['Contract'].map(contract_mapping)

# 2. One-Hot Encoding for Nominal Categories
# Defining columns that lack a natural order
nominal_cols = [
    'gender', 
    'PhoneService', 
    'InternetService', 
    'PaperlessBilling', 
    'PaymentMethod'
]

# Apply one-hot encoding and drop the first dummy to avoid multicollinearity
X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

# Ensure resulting dummy columns are integers (if pandas outputs booleans)
X = X.astype({col: int for col in X.select_dtypes(include='bool').columns})

In [53]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Train-Test Split (80% Training, 20% Testing)
# Splitting X, y_reg, and y_clf simultaneously ensures indices align perfectly
X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, 
    y_reg, 
    y_clf, 
    test_size=0.2, 
    random_state=42
)

# 2. Initialize the StandardScaler
scaler = StandardScaler()

# 3. Fit and transform on the TRAINING data ONLY
# This calculates the mean and standard deviation of the training set
X_train_scaled = scaler.fit_transform(X_train)

# 4. Transform the TEST data
# This scales the test set using the mean and standard deviation learned from the training set
X_test_scaled = scaler.transform(X_test)

#  Convert the scaled arrays back to pandas DataFrames for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

# 1. Train OLS Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_reg_train)

# 2. Predict and Evaluate OLS
y_pred_reg_lr = lr.predict(X_test_scaled)
mse_lr = mean_squared_error(y_reg_test, y_pred_reg_lr)
r2_lr = r2_score(y_reg_test, y_pred_reg_lr)

print("--- OLS Linear Regression ---")
print(f"MSE: {mse_lr:.4f}")
print(f"R²: {r2_lr:.4f}\n")

# 3. Extract and display coefficients
coef_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr.coef_})
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)

print("Coefficients:")
print(coef_df[['Feature', 'Coefficient']].to_string(index=False))

# Identify Top 3 features by absolute magnitude
print("\nTop 3 Features by Absolute Coefficient:")
print(coef_df.head(3)[['Feature', 'Coefficient', 'Abs_Coefficient']].to_string(index=False))

# 4. Train Ridge Regression (alpha=1.0)
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_reg_train)

# 5. Predict and Evaluate Ridge
y_pred_reg_ridge = ridge.predict(X_test_scaled)
mse_ridge = mean_squared_error(y_reg_test, y_pred_reg_ridge)
r2_ridge = r2_score(y_reg_test, y_pred_reg_ridge)

print("\n--- Ridge Regression ---")
print(f"MSE: {mse_ridge:.4f}")
print(f"R²: {r2_ridge:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# 1. Check Class Imbalance
class_proportions = y_clf_train.value_counts(normalize=True)
print(class_proportions)

# 2. Initialize and Train Logistic Regression
# Because the minority class is < 35%, we apply class_weight='balanced'
lr1 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr1.fit(X_train_scaled, y_clf_train)

# 3. Predict Class Labels and Probabilities
y_pred_clf = lr1.predict(X_test_scaled)
y_pred_probs = lr1.predict_proba(X_test_scaled)[:, 1]

# 4. Generate Classification Metrics
cm = confusion_matrix(y_clf_test, y_pred_clf)
cr = classification_report(y_clf_test, y_pred_clf)

print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", cr)

# 5. Calculate AUC and Plot ROC Curve
auc_score = roc_auc_score(y_clf_test, y_pred_probs)
print(f"\nAUC Score: {auc_score:.4f}")

fpr, tpr, thresholds = roc_curve(y_clf_test, y_pred_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='#1f77b4', linewidth=2)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.annotate(f'AUC = {auc_score:.3f}', xy=(0.6, 0.2), xytext=(0.6, 0.2), fontsize=14,
             bbox=dict(boxstyle="round,pad=0.3", edgecolor='black', facecolor='white'))
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_pred_probs = lr1.predict_proba(X_test_scaled)[:, 1]

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

print("Threshold | Precision | Recall | F1")
print("---|---|---|---")
for t in thresholds:
    # Convert probabilities to class labels based on the current threshold
    y_pred_t = (y_pred_probs >= t).astype(int)
    
    # Calculate metrics
    p = precision_score(y_clf_test, y_pred_t, zero_division=0)
    r = recall_score(y_clf_test, y_pred_t, zero_division=0)
    f1 = f1_score(y_clf_test, y_pred_t, zero_division=0)
    
    # Print row
    print(f"{t:.2f} | {p:.4f} | {r:.4f} | {f1:.4f}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, roc_auc_score

# 1. Baseline Model (Default C=1.0)
lr_base = LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)
lr_base.fit(X_train_scaled, y_clf_train)

y_pred_base = lr_base.predict(X_test_scaled)
y_prob_base = lr_base.predict_proba(X_test_scaled)[:, 1]

# 2. Experimental Model (Strong Regularization C=0.01)
lr_reg = LogisticRegression(C=0.01, class_weight='balanced', max_iter=1000, random_state=42)
lr_reg.fit(X_train_scaled, y_clf_train)

y_pred_reg = lr_reg.predict(X_test_scaled)
y_prob_reg = lr_reg.predict_proba(X_test_scaled)[:, 1]

# 3. Calculate and Compare Metrics
metrics = {
    'C=1.0 (Baseline)': [
        precision_score(y_clf_test, y_pred_base), 
        recall_score(y_clf_test, y_pred_base), 
        roc_auc_score(y_clf_test, y_prob_base)
    ],
    'C=0.01 (Strong Reg)': [
        precision_score(y_clf_test, y_pred_reg), 
        recall_score(y_clf_test, y_pred_reg), 
        roc_auc_score(y_clf_test, y_prob_reg)
    ]
}

print("Metrics Comparison:", metrics)

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score


n_bootstraps = 500
auc_diffs = []
y_test_arr = y_clf_test.values
rng = np.random.RandomState(42) 

for _ in range(n_bootstraps):
    # Sample row indices with replacement
    indices = rng.choice(len(y_test_arr), size=len(y_test_arr), replace=True)
    
    y_true_boot = y_test_arr[indices]
    
    # Skip iteration if the bootstrap sample only contains one class (AUC requires both)
    if len(np.unique(y_true_boot)) < 2:
        continue
        
    # Index predicted probabilities using the same sampled indices
    y_prob_base_boot = y_prob_base[indices]
    y_prob_reg_boot = y_prob_reg[indices]
    
    # Compute AUC for both models on the bootstrap sample
    auc_base_boot = roc_auc_score(y_true_boot, y_prob_base_boot)
    auc_reg_boot = roc_auc_score(y_true_boot, y_prob_reg_boot)
    
    # Compute and store the difference (C=1.0 - C=0.01)
    auc_diffs.append(auc_base_boot - auc_reg_boot)

# Calculate the mean and the 95% Confidence Interval (2.5th and 97.5th percentiles)
mean_diff = np.mean(auc_diffs)
ci_lower = np.percentile(auc_diffs, 2.5)
ci_upper = np.percentile(auc_diffs, 97.5)

print(f"Mean AUC Difference: {mean_diff:.5f}")
print(f"95% CI: [{ci_lower:.5f}, {ci_upper:.5f}]")